# 07 - Discovered indicators

The indicators the network learned, drawn on the price of one window next to the same indicators at their
textbook periods: the discovered indicators are what a run delivers (VISION), and the prediction and trading
quality in 01-04 are the evidence that they are good. Each copy of each family (moving average, Bollinger bands,
RSI, MACD; the copies start from `MA_SPANS`, `BB_PERIODS`, `RSI_PERIODS` and `MACD_SETTINGS`) is drawn twice on the
same window: solid at the period the served model applied to that window, dashed at its configured textbook
period. The lines use the model's own formulas (an EWMA with alpha = 2 / (period + 1), started at the window's first
bar), so they are the channels the network reads, in price units.

Set `WINDOW` to a window index, or to "typical" (closest to the block's median periods), "longest" / "shortest"
(where the per-window adjustment stretches or shrinks the periods most) or "last"; `BLOCK` to another block of
the run's fold. Nothing here writes to the run.

In [1]:
# Parameters
RUN_DIR = None                  # a runs/<id> directory; None -> the newest run under RUNS_DIR
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
BLOCK = "test"                  # "train", "val", "cal" or "test"
WINDOW = "typical"              # a window index, or "typical" / "longest" / "shortest" / "last"

In [2]:
from neural_trade.data.processor import split_arrays
from neural_trade.notebook import pick_run
from neural_trade.registries.visualizations import Visualizations
from neural_trade.serving.predictor import Predictor
from neural_trade.visualization import analytics_tables as AT
from neural_trade.visualization.discovered_indicators import discovered_table
from neural_trade.visualization.indicator_evolution import applied_periods

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
predictor = Predictor.from_artifacts(run_dir / "artifacts")
cfg = predictor.config.copy(CSV_PATH=CSV_PATH)
blocks = split_arrays(cfg)
block = blocks[BLOCK]
times = blocks["df"]["timestamp"].to_numpy()[block["anchor_bar"]]
metrics = run_dir / "metrics.jsonl"
applied = applied_periods(predictor, block["X"], block=BLOCK)   # the period each window gets (served weights)
print(f"{len(block['X']):,} {BLOCK} windows of {cfg.LOOKBACK} bars; served weights: epoch",
      predictor.bundle.meta.get("weights_epoch"))

run: ..\runs\20260929T081632Z-426de4f-dirty-aba344d6
CalibrationPipeline loaded from '..\runs\20260929T081632Z-426de4f-dirty-aba344d6\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


7,236 test windows of 60 bars; served weights: epoch 19


## The learned indicators on price

Top: the block's close, one point per window, with the drawn window shaded. The grid: one row per family, one
column per copy; each panel's heading gives the learned and the textbook period (MACD: fast / slow / signal).
Below it, each base period over training (from the recorded start, the dashed line is its textbook value) and,
right of the last epoch, how the period the model applies varies across the block's windows: 5-95% and the middle
50% of the windows, their median, this window's period and the served base period. A period longer than the
window has not warmed up by the window's end (the EWMA starts at the window's first bar); the table's
"applied > lookback" column counts those windows.

In [3]:
Visualizations.build("discovered_indicators", block["X"], cfg, applied=applied, metrics=metrics, window=WINDOW,
                     times=times).show()

## Learned against textbook periods

The table behind the figure, in bars: the served base period (the trained logit alone) and the median applied
period, each with its change against the textbook period; the applied range over the block's windows; this
window's period; the range the base period took during training; the share of windows whose applied period is
longer than the window; and whether the base period sits at a clip bound.

In [4]:
AT.styled(discovered_table(applied, cfg, metrics=metrics, window=WINDOW), digits=2)

,indicator,textbook,learned (served base),base vs textbook %,applied median,median vs textbook %,applied p5,applied p95,this window,"applied > lookback, % of windows",training min,training max,base near a clip bound
period,,,,,,,,,,,,,
ma_period_0,MA #0,5.00,3.83,-23.40,3.72,-25.51,3.01,4.71,3.72,0.00,3.74,5.00,
ma_period_1,MA #1,10.00,9.37,-6.31,11.39,13.94,10.07,13.42,11.25,0.00,9.05,10.52,
ma_period_2,MA #2,30.00,15.84,-47.21,18.15,-39.49,14.63,23.81,18.23,0.00,15.21,30.00,
macd_0_fast,MACD #0 fast,12.00,5.74,-52.19,4.61,-61.59,3.92,6.03,4.59,0.00,5.59,12.00,
macd_0_slow,MACD #0 slow,26.00,30.87,18.74,24.56,-5.52,20.54,28.55,24.87,0.00,26.00,33.45,
macd_0_signal,MACD #0 signal,9.00,6.53,-27.48,6.66,-25.97,6.26,7.81,6.70,0.00,6.36,9.00,
macd_1_fast,MACD #1 fast,5.00,2.34,-53.19,2.06,-58.87,1.83,2.60,2.05,0.00,2.31,5.00,
macd_1_slow,MACD #1 slow,35.00,59.09,68.82,73.02,108.62,52.10,94.32,73.70,79.81,35.00,59.90,ceiling
macd_1_signal,MACD #1 signal,5.00,5.85,17.03,6.05,20.97,5.59,6.75,6.06,0.00,4.43,6.22,
